## Gold Layer — Analytics-Ready Aggregates
**Reads from:** `main.silver.*`
**Writes to:** `main.gold.*`

Tables produced:
- `main.gold.ticker_daily_summary`   — price + sentiment per ticker per day
- `main.gold.sector_rankings`        — sector-level market cap + performance
- `main.gold.sentiment_summary`      — sentiment distribution per ticker
- `main.gold.top_movers`             — best and worst performing tickers


In [ ]:
# 0. Imports and config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

PROCESSED_AT = datetime.now().isoformat()
print(f"Gold aggregation started: {PROCESSED_AT}")

# Verify Silver columns available
print("\nSilver companies columns:")
spark.table("main.silver.companies").printSchema()


In [ ]:
# 1. Setup Gold schema
spark.sql("CREATE SCHEMA IF NOT EXISTS main.gold")
print("Schema main.gold ready")


In [ ]:
# 2. Gold — Ticker Daily Summary
print("\n--- Building ticker_daily_summary ---")

silver_prices_all= spark.table("main.silver.price_snapshots")
silver_companies = spark.table("main.silver.companies")
silver_news      = spark.table("main.silver.news_articles")

# Keep only LATEST snapshot_date per ticker
# Silver retains all historical dates — Gold needs only the most recent
from pyspark.sql.window import Window as W2
latest_window = W2.partitionBy("ticker").orderBy(F.col("snapshot_date").desc())
silver_prices = (
    silver_prices_all
    .withColumn("_rn", F.row_number().over(latest_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
print(f"Silver prices (latest per ticker): {silver_prices.count()} rows")

# Use sic_description as sector proxy (Polygon.io doesn't return sector directly)
companies_slim = silver_companies.select(
    "ticker", "name", "exchange_name", "market_cap_billions",
    F.col("sic_description").alias("sector")
)

# Sentiment score per ticker
# positive=1, neutral=0, negative=-1
sentiment_per_ticker = (
    silver_news
    .withColumn("sentiment_score",
        F.when(F.col("sentiment") == "positive",  1.0)
         .when(F.col("sentiment") == "negative", -1.0)
         .otherwise(0.0)
    )
    .groupBy("ticker")
    .agg(
        F.count("article_id").alias("news_count"),
        F.round(F.avg("sentiment_score"), 4).alias("avg_sentiment_score"),
        F.sum(F.when(F.col("sentiment") == "positive", 1).otherwise(0)).alias("positive_count"),
        F.sum(F.when(F.col("sentiment") == "negative", 1).otherwise(0)).alias("negative_count"),
        F.sum(F.when(F.col("sentiment") == "neutral",  1).otherwise(0)).alias("neutral_count")
    )
)

# Join price + company + sentiment
ticker_daily_summary = (
    silver_prices
    .join(companies_slim, on="ticker", how="left")
    .join(sentiment_per_ticker, on="ticker", how="left")
    .withColumn("processed_at", F.lit(PROCESSED_AT))
    .select(
        "ticker", "name", "exchange_name", "sector",
        "market_cap_billions", "snapshot_date",
        "open", "high", "low", "close", "volume",
        "vwap", "daily_return_pct", "price_range", "is_up_day",
        "news_count", "avg_sentiment_score",
        "positive_count", "negative_count", "neutral_count",
        "processed_at"
    )
)

(ticker_daily_summary
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.gold.ticker_daily_summary"))

count = spark.table("main.gold.ticker_daily_summary").count()
print(f"Written {count} rows → main.gold.ticker_daily_summary ✓")

print("\nSample:")
spark.table("main.gold.ticker_daily_summary") \
     .select("ticker", "name", "close", "daily_return_pct",
             "is_up_day", "news_count", "avg_sentiment_score") \
     .orderBy(F.col("daily_return_pct").desc()) \
     .show(5, truncate=False)


In [ ]:
# 3. Gold — Sector Rankings
print("\n--- Building sector_rankings ---")

sector_rankings = (
    spark.table("main.gold.ticker_daily_summary")
    .groupBy("sector")
    .agg(
        F.count("ticker").alias("ticker_count"),
        F.round(F.sum("market_cap_billions"), 2).alias("total_market_cap_billions"),
        F.round(F.avg("market_cap_billions"), 2).alias("avg_market_cap_billions"),
        F.round(F.avg("daily_return_pct"), 4).alias("avg_daily_return_pct"),
        F.round(F.avg("avg_sentiment_score"), 4).alias("avg_sentiment_score"),
        F.sum("news_count").alias("total_news_count"),
        F.collect_list("ticker").alias("tickers")
    )
    .withColumn("sector_rank",
        F.rank().over(
            Window.orderBy(F.col("total_market_cap_billions").desc())
        )
    )
    .withColumn("processed_at", F.lit(PROCESSED_AT))
    .orderBy("sector_rank")
)

(sector_rankings
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.gold.sector_rankings"))

count = spark.table("main.gold.sector_rankings").count()
print(f"Written {count} rows → main.gold.sector_rankings ✓")

print("\nSample:")
spark.table("main.gold.sector_rankings") \
     .select("sector_rank", "sector", "ticker_count",
             "total_market_cap_billions", "avg_daily_return_pct",
             "avg_sentiment_score") \
     .show(10, truncate=False)


In [ ]:
# 4. Gold — Sentiment Summary
print("\n--- Building sentiment_summary ---")

sentiment_summary = (
    spark.table("main.gold.ticker_daily_summary")
    .select("ticker", "name", "close", "daily_return_pct",
            "news_count", "avg_sentiment_score",
            "positive_count", "negative_count", "neutral_count")
    .withColumn("sentiment_signal",
        F.when(F.col("avg_sentiment_score") >  0.3, "BULLISH")
         .when(F.col("avg_sentiment_score") < -0.3, "BEARISH")
         .otherwise("NEUTRAL")
    )
    .withColumn("sentiment_confidence",
        F.when(F.col("news_count") >= 8, "HIGH")
         .when(F.col("news_count") >= 4, "MEDIUM")
         .otherwise("LOW")
    )
    .withColumn("processed_at", F.lit(PROCESSED_AT))
    .orderBy(F.col("avg_sentiment_score").desc())
)

(sentiment_summary
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.gold.sentiment_summary"))

count = spark.table("main.gold.sentiment_summary").count()
print(f"Written {count} rows → main.gold.sentiment_summary ✓")

print("\nSample:")
spark.table("main.gold.sentiment_summary") \
     .select("ticker", "close", "news_count",
             "avg_sentiment_score", "sentiment_signal", "sentiment_confidence") \
     .show(5, truncate=False)


In [ ]:
# 5. Gold — Top Movers
print("\n--- Building top_movers ---")

window_return = Window.orderBy(F.col("daily_return_pct").desc())

top_movers = (
    spark.table("main.gold.ticker_daily_summary")
    .withColumn("return_rank", F.rank().over(window_return))
    .withColumn("mover_type",
        F.when(F.col("daily_return_pct") > 0, "GAINER")
         .when(F.col("daily_return_pct") < 0, "LOSER")
         .otherwise("FLAT")
    )
    .select(
        "return_rank", "ticker", "name", "sector",
        "close", "daily_return_pct", "price_range",
        "volume", "mover_type", "avg_sentiment_score",
        "snapshot_date", "processed_at"
    )
    .orderBy("return_rank")
)

(top_movers
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.gold.top_movers"))

count = spark.table("main.gold.top_movers").count()
print(f"Written {count} rows → main.gold.top_movers ✓")

print("\nTop Gainers:")
spark.table("main.gold.top_movers") \
     .filter(F.col("mover_type") == "GAINER") \
     .select("return_rank", "ticker", "close",
             "daily_return_pct", "avg_sentiment_score") \
     .show(5)

print("Top Losers:")
spark.table("main.gold.top_movers") \
     .filter(F.col("mover_type") == "LOSER") \
     .select("return_rank", "ticker", "close",
             "daily_return_pct", "avg_sentiment_score") \
     .show(5)


In [ ]:
# 6. Gold Summary
print("\n=== Gold Aggregation Summary ===")
print(f"Processed at: {PROCESSED_AT}\n")

gold_tables = {
    "ticker_daily_summary": "Price + sentiment per ticker per day",
    "sector_rankings"     : "Market cap + performance by sector",
    "sentiment_summary"   : "Sentiment signal and confidence per ticker",
    "top_movers"          : "Best and worst performers by daily return"
}

for table, desc in gold_tables.items():
    try:
        count = spark.table(f"main.gold.{table}").count()
        print(f"  main.gold.{table:<25} rows: {count:>4}  — {desc}")
    except Exception as e:
        print(f"  main.gold.{table:<25} ERROR: {e}")

print("\nMedallion Architecture complete:")
print("  Bronze (raw)   → main.bronze.*  append-only, full raw_json")
print("  Silver (clean) → main.silver.*  deduplicated, typed, enriched")
print("  Gold   (agg)   → main.gold.*    analytics-ready aggregates")
print("\nGold aggregation complete ✓")
